# Sensibilidad MLP de una capa oculta

Esta corrida cambia **una sola cosa** respecto del desarrollo MLP primario: el predictor pasa de `32→64→64→32` a la lectura alternativa del paper `32→64→32`. Usa únicamente train y validation; no construye test.

## Pregunta

¿La arquitectura de una capa oculta produce checkpoints predictivos estables en las cinco seeds y, si lo hace, alcanza clustering útil? Los umbrales no se modificaron después del resultado anterior.

In [ ]:
# ruff: noqa: E402, E501
import json
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import (
    PaperCheckpointReplayConfig,
    load_paper_mlp_one_hidden_development_config,
)
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_evaluation import (
    evaluate_paper_mlp_clustering_gate,
    evaluate_paper_mlp_seed_clustering,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    evaluate_scale_invariant_seed_stability_gate,
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    summarize_seed_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
torch.use_deterministic_algorithms(True)

In [ ]:
config_path = ROOT / "configs" / "paper_mlp_one_hidden_development.yaml"
config = load_paper_mlp_one_hidden_development_config(config_path)
replay_config = PaperCheckpointReplayConfig(metric_absolute_tolerance=1e-8)
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {validation_dataset.sample_key(index) for index in range(len(validation_dataset))}
assert config.model.mlp_depth == "one_hidden"
assert len(train_dataset) == 64 * len(PAPER_REGIME_NAMES) == 1152
assert len(validation_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert train_keys.isdisjoint(validation_keys)
print(json.dumps(asdict(config), indent=2))
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}. Test no fue instanciado.")

In [ ]:
def closest_checkpoint_row(history):
    initial = history[0]
    rows = []
    for row in history[1:]:
        loss_ratio = row.validation_loss / max(initial.validation_loss, 1e-12)
        gap = row.validation_loss / max(row.train_loss, 1e-12)
        spread_ratio = row.validation_embedding_std / max(initial.validation_embedding_std, 1e-12)
        checks = (
            loss_ratio <= config.checkpoint_gate.max_validation_loss_ratio,
            gap <= config.checkpoint_gate.max_validation_train_loss_ratio,
            spread_ratio >= config.checkpoint_gate.min_validation_embedding_std_ratio,
            row.validation_effective_rank >= config.checkpoint_gate.min_validation_effective_rank,
        )
        rows.append({
            "epoch": row.epoch, "validation_loss": row.validation_loss,
            "loss_ratio": loss_ratio, "gap": gap, "spread_ratio": spread_ratio,
            "rank": row.validation_effective_rank, "passed_count": sum(checks),
        })
    return min(rows, key=lambda item: (-item["passed_count"], item["validation_loss"], item["epoch"]))

@torch.no_grad()
def validation_embeddings(model, run_config):
    device = torch.device(run_config.device)
    model.to(device).eval()
    embedding_chunks, label_chunks = [], []
    for context, _, labels in make_paper_loader(validation_dataset, run_config, shuffle=False):
        embedding_chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
        label_chunks.append(labels.numpy())
    return np.concatenate(embedding_chunks), np.concatenate(label_chunks)

print("Funciones listas.")

In [ ]:
models, histories, diagnostics = {}, {}, {}
summaries, replay_results, times = [], {}, {}
for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)
    started = time.perf_counter()
    run = run_paper_train_validation_with_checkpoint(
        model, train_dataset, validation_dataset, run_config, config.checkpoint_gate
    )
    times[seed] = time.perf_counter() - started
    histories[seed] = run.history
    diagnostics[seed] = closest_checkpoint_row(run.history)
    if run.selection is None:
        row = diagnostics[seed]
        print(f"seed={seed} NONE closest={row['epoch']} checks={row['passed_count']}/4 rank={row['rank']:.2f}")
        continue
    replay = verify_paper_checkpoint_replay(
        model, run, validation_dataset, run_config, replay_config, expected_epoch=run.selection.epoch
    )
    models[seed], replay_results[seed] = model, replay
    summaries.append(summarize_seed_checkpoint(seed, run.selection))
    print(
        f"seed={seed} epoch={run.selection.epoch} val/base={run.selection.validation_loss_ratio:.3f} "
        f"rank={run.selection.validation_effective_rank:.2f} replay={'PASS' if replay.passed else 'FAIL'}"
    )
replay_passed = set(replay_results) == set(config.sweep.seeds) and all(row.passed for row in replay_results.values())
predictive_gate = evaluate_scale_invariant_seed_stability_gate(summaries, config.sweep, config.stability_gate)
predictive_passed = replay_passed and predictive_gate.passed
print(f"Predictivo: {'PASS' if predictive_passed else 'FAIL'}; tiempo={sum(times.values()):.1f}s")
print(json.dumps(asdict(predictive_gate), indent=2))

In [ ]:
clustering_metrics, clustering_gate = [], None
if predictive_passed:
    reference_labels = None
    for seed in config.sweep.seeds:
        embeddings, labels = validation_embeddings(models[seed], replace(config.train, seed=seed))
        reference_labels = labels if reference_labels is None else reference_labels
        assert np.array_equal(labels, reference_labels)
        clustering_metrics.append(
            evaluate_paper_mlp_seed_clustering(embeddings, labels, seed, config.clustering)
        )
    clustering_gate = evaluate_paper_mlp_clustering_gate(
        clustering_metrics, config.sweep, config.clustering_gate
    )
    print(json.dumps(asdict(clustering_gate), indent=2))
else:
    print("Clustering omitido: falló el prerrequisito predictivo.")
development_passed = predictive_passed and clustering_gate is not None and clustering_gate.passed
print(f"Resultado conjunto: {'PASS' if development_passed else 'FAIL'}")
print("Test no fue construido ni consultado.")

In [ ]:
summary_by_seed = {row.seed: row for row in summaries}
cluster_by_seed = {row.seed: row for row in clustering_metrics}
print("seed epoch val/base gap spread rank purity_mean purity_sd")
for seed in config.sweep.seeds:
    summary, diagnostic, cluster = summary_by_seed.get(seed), diagnostics[seed], cluster_by_seed.get(seed)
    epoch = str(summary.checkpoint_epoch) if summary else "NONE"
    purity = f"{cluster.mean_purity:.2%}" if cluster else "not-run"
    purity_sd = f"{cluster.purity_std:.2%}" if cluster else "not-run"
    print(
        f"{seed:>4d} {epoch:>5s} {diagnostic['loss_ratio']:>8.3f} {diagnostic['gap']:>5.3f} "
        f"{diagnostic['spread_ratio']:>6.3f} {diagnostic['rank']:>5.2f} {purity:>11s} {purity_sd:>9s}"
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(config.sweep.seeds)))
for color, seed in zip(colors, config.sweep.seeds, strict=True):
    history = histories[seed]
    epochs = np.array([row.epoch for row in history])
    losses = np.array([row.validation_loss for row in history])
    ranks = np.array([row.validation_effective_rank for row in history])
    axes[0].plot(epochs, losses / losses[0], marker="o", markersize=3, color=color, label=f"seed {seed}")
    axes[1].plot(epochs, ranks, marker="o", markersize=3, color=color)
axes[0].axhline(config.checkpoint_gate.max_validation_loss_ratio, color="tab:red", linestyle="--")
axes[0].set(title="Error validation / inicial", xlabel="Época", ylabel="Ratio")
axes[0].legend(fontsize=8)
axes[1].axhline(config.checkpoint_gate.min_validation_effective_rank, color="tab:red", linestyle="--")
axes[1].set(title="Rango efectivo", xlabel="Época", ylabel="Rango")
if clustering_gate is not None:
    seeds = np.array(config.sweep.seeds)
    means = np.array([cluster_by_seed[seed].mean_purity for seed in seeds])
    stds = np.array([cluster_by_seed[seed].purity_std for seed in seeds])
    axes[2].errorbar(seeds, means, yerr=stds, fmt="o", capsize=5)
    axes[2].axhline(config.clustering_gate.min_worst_seed_mean_purity, color="tab:red", linestyle="--")
    axes[2].axhline(0.6548, color="tab:green", linestyle=":", label="paper")
    axes[2].set(title="Pureza media ± sd", xlabel="Seed", ylabel="Pureza")
    axes[2].legend()
else:
    axes[2].axis("off")
    axes[2].text(0.05, 0.9, "Clustering omitido", va="top", fontsize=14)
plt.show()

if clustering_gate is None:
    conclusion = "La variante no habilitó clustering porque no todas las seeds pasaron el gate predictivo."
    clustering_text = "No medido."
else:
    conclusion = (
        "La variante pasa desarrollo y queda lista para congelar una evaluación test."
        if development_passed else "La variante llega a clustering pero no pasa el gate de pureza/estabilidad."
    )
    clustering_text = (
        f"Pureza global `{clustering_gate.overall_mean_purity:.2%}`, peor seed "
        f"`{clustering_gate.worst_seed_mean_purity:.2%}`, CV "
        f"`{clustering_gate.seed_mean_purity_coefficient_of_variation:.3f}`."
    )
display(Markdown(f"""## Veredicto

- **Predictivo:** {'PASS' if predictive_passed else 'FAIL'}.
- **Clustering:** {clustering_text}
- **Conjunto:** {'PASS' if development_passed else 'FAIL'}.

{conclusion} Test no fue construido ni consultado.
"""))